In [1]:
from utils.explainer import NaturalLanguageExplainer

In [2]:
explainer = NaturalLanguageExplainer(max_length=256)

Using device: mps
Startup time: 0.6902363300323486


In [3]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."

In [4]:
explainer.get_explanation(text)

'This text is classified as not anonymized because it contains specific personal information about a person named Stephen J. Gordon, including his full name, date of birth, place of birth, achievements in chess, and details about his career and accomplishments. The use of specific names, dates, and locations make it possible to identify the individual being discussed, therefore it is not anonymized.'

In [5]:
from utils.read_jsonl import read_jsonl

eval_df = read_jsonl("../DB-bio/combined_val_and_val_sft_anonymized.jsonl")

In [6]:
eval_df = eval_df[eval_df["text"].apply(len) < 1100]
len(eval_df)

131

In [7]:
encoded = eval_df["text"].apply(lambda t: explainer.feature_importance.tokenizer.encode(t,truncation=True, max_length=256, padding="max_length"))
encoded = encoded.apply(lambda x: 1 if 0 in x else 0)
eval_df = eval_df[encoded == 1]
len(eval_df)

124

In [10]:
def get_natural_language_explanation(row):
    true_label = row["label"]
    
    print(row.name, true_label)
    _text = row["text"]
    explanation = explainer.get_explanation(_text)
    print(explanation)
    return explanation

In [11]:
len(eval_df)

124

In [ ]:
eval_df["explanation"] = eval_df.apply(get_natural_language_explanation,axis=1)

In [17]:
eval_df.to_csv("../4_ExplanationResults/eval_df_with_explations_new.csv", index=False)

In [41]:
eval_df.head(1)["explanation"]

0    This text is classified as not anonymized beca...
Name: explanation, dtype: object

In [42]:
eval_df.tail(1)["explanation"]

485    This text is classified as anonymized because ...
Name: explanation, dtype: object